<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-05 22:44:05
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46


=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.53 C
CAGR/XIRR %:     1.06%
Est FTT Value:   2.74 C
Future Upside:   1.21 C (79.45%)
=================== PERFORMANCE ==========================
Current PnL:             -43.73 L (-22.78%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    4.86 L (3.26%)
CY Booked PnL:           5.71 L (3.74%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               2.39 L (1.59%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             2.05 L
Total Loss:               -45.78 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,0.41,25.80,7.71,35.51,20328.0,54079.0,263653.0,1285.01,24.82,55.0,H-LC,11.55,120.0,2.66,1.72,38.75,AR,ATH,PHARMA
76,TTKPRESTIG,0.70,-28.37,39.73,0.09,28680.0,-28589.0,72188.0,770.00,-12.70,59.0,M-SC,4.35,250.0,-1.00,0.47,29.93,OX40N,NTT,DURABLES
40,ICICIPRULI,3.26,-29.99,23.47,-13.56,32368.0,-59091.0,137914.0,522.35,-55.17,40.0,X-MC,9.39,78.0,-1.83,0.90,3.94,X40,ATH,INSURANCE
12,BOSCH-HCIL,-2.33,-26.39,35.84,-0.01,33716.0,-33731.0,94074.0,2282.00,732.37,43.0,M-SC,19.93,231.0,-1.00,0.62,60.13,OX40N,BTT,AC
66,SIS,0.63,-10.55,35.56,21.26,35319.0,-11710.0,99322.0,528.00,91.06,48.0,H-SC,8.05,168.0,-0.33,0.65,47.68,OX40N,NTT,MISC


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,2.81,9.71,31.20,43.94,86028.0,24398.0,275730.0,2782.73,13.61,66.0,X-LC,5.72,29.0,0.28,1.80,26.60,X40N,ATH,AUTO
52,MARUTI,5.48,-7.56,35.32,25.08,98311.0,-22774.0,278344.0,17120.06,-13.15,70.0,X-LC,13.62,33.0,-0.23,1.82,13.27,XR,ATH,AUTO
51,M&M,5.59,-2.36,20.32,17.48,59738.0,-7090.0,293985.0,3762.88,-14.64,68.0,X-LC,9.31,25.0,-0.12,1.92,13.68,X40N,ATH,AUTO
35,HDFCLIFE,2.12,-5.80,49.21,40.56,144238.0,-18051.0,293108.0,811.41,-31.42,56.0,X-LC,10.08,31.0,-0.13,1.92,5.88,X40,ATH,INSURANCE
23,DMART,0.18,-16.00,57.64,32.43,171502.0,-56660.0,297540.0,5391.45,-30.13,52.0,X-LC,16.19,14.0,-0.33,1.95,9.13,X40N,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
53,MASFIN,7.65,-16.67,45.51,21.26,37158.0,-16332.0,81648.0,396.02,-35.83,56.0,H-SC,14.94,167.0,-0.44,0.53,11.24,XR,ATH,FINANCE
51,M&M,5.59,-2.36,20.32,17.48,59738.0,-7090.0,293985.0,3762.88,-14.64,68.0,X-LC,9.31,25.0,-0.12,1.92,13.68,X40N,ATH,AUTO
52,MARUTI,5.48,-7.56,35.32,25.08,98311.0,-22774.0,278344.0,17120.06,-13.15,70.0,X-LC,13.62,33.0,-0.23,1.82,13.27,XR,ATH,AUTO
74,TMPV,5.46,-28.51,104.29,46.05,333560.0,-127533.0,319839.0,600.00,-85.99,62.0,X-LC,17.51,4.0,-0.38,2.09,12.44,XY24,NTT,AUTO
44,INDUSINDBK,5.35,-32.71,97.49,32.89,50649.0,-25254.0,51953.0,1800.00,282.48,58.0,L-MC,6.78,258.0,-0.50,0.34,21.33,XR,NTT,BANKS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
80,VALIANTORG,-14.84,-23.24,126.89,74.15,159347.0,-38026.0,125579.0,838.00,NaN,38.0,NaN,46.15,145.0,-0.24,0.82,86.88,XR,NTT,CHEMICALS
20,DABUR,-2.53,-24.73,90.91,43.70,169752.0,-61343.0,186725.0,735.00,-35.11,54.0,X-MC,12.88,70.0,-0.36,1.22,7.60,XY24,BTT,FMCG
12,BOSCH-HCIL,-2.33,-26.39,35.84,-0.01,33716.0,-33731.0,94074.0,2282.00,732.37,43.0,M-SC,19.93,231.0,-1.00,0.62,60.13,OX40N,BTT,AC
1,ABBOTINDIA,-1.90,-10.75,29.26,15.37,46708.0,-19225.0,159630.0,34390.26,-27.37,55.0,X-MC,1.33,58.0,-0.41,1.04,8.13,X40,ATH,PHARMA
77,UNITDSPR,-1.75,2.44,17.89,20.77,42910.0,5710.0,239854.0,1644.00,-12.73,56.0,X-MC,4.06,64.0,0.13,1.57,15.30,X40N,NTT,BREWERIES


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.48,48.73,38.80,106.44,85202.0,71950.0,219594.0,1629.94,8.16,41.0,M-SC,25.42,220.0,0.84,1.44,66.39,OX40N,ATH,CABLES
66,SIS,0.63,-10.55,35.56,21.26,35319.0,-11710.0,99322.0,528.00,91.06,48.0,H-SC,8.05,168.0,-0.33,0.65,47.68,OX40N,NTT,MISC
12,BOSCH-HCIL,-2.33,-26.39,35.84,-0.01,33716.0,-33731.0,94074.0,2282.00,732.37,43.0,M-SC,19.93,231.0,-1.00,0.62,60.13,OX40N,BTT,AC
76,TTKPRESTIG,0.70,-28.37,39.73,0.09,28680.0,-28589.0,72188.0,770.00,-12.70,59.0,M-SC,4.35,250.0,-1.00,0.47,29.93,OX40N,NTT,DURABLES
49,KANSAINER,4.57,-31.84,66.47,13.47,122182.0,-85851.0,183816.0,340.00,-28.68,68.0,H-SC,8.07,162.0,-0.70,1.20,29.92,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.48,48.73,38.80,106.44,85202.0,71950.0,219594.0,1629.94,8.16,41.0,M-SC,25.42,220.0,0.84,1.44,66.39,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.41,25.80,7.71,35.51,20328.0,54079.0,263653.0,1285.01,24.82,55.0,H-LC,11.55,120.0,2.66,1.72,38.75,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.48,48.73,38.80,106.44,85202.0,71950.0,219594.0,1629.94,8.16,41.0,M-SC,25.42,220.0,0.84,1.44,66.39,OX40N,ATH,CABLES
67,STARHEALTH,2.23,4.95,40.32,47.26,118959.0,13915.0,295038.0,761.00,-7.47,51.0,H-SC,5.07,173.0,0.12,1.93,27.57,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.48,48.73,38.80,106.44,85202.0,71950.0,219594.0,1629.94,8.16,41.0,M-SC,25.42,220.0,0.84,1.44,66.39,OX40N,ATH,CABLES
84,WIPRO,1.98,-8.82,111.69,93.01,254832.0,-22080.0,228160.0,420.00,-23.12,69.0,M-LC,8.03,98.0,-0.09,1.49,26.54,XR,NTT,IT
70,TANLA,-0.51,-25.89,190.34,115.17,381936.0,-70101.0,200660.0,1903.93,NaN,76.0,NaN,12.81,147.0,-0.18,1.31,77.38,AR,ATH,IT
80,VALIANTORG,-14.84,-23.24,126.89,74.15,159347.0,-38026.0,125579.0,838.00,NaN,38.0,NaN,46.15,145.0,-0.24,0.82,86.88,XR,NTT,CHEMICALS
42,INDIAMART,-0.94,-32.26,196.62,100.92,164264.0,-39792.0,83544.0,4675.60,NaN,47.0,NaN,16.71,140.0,-0.24,0.55,5.99,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,1.71,-21.38,91.93,50.88,139366.0,-41238.0,151600.0,58192.80,-38.62,38.0,X-SC,6.67,90.0,-0.30,0.99,15.50,X40N,ATH,ELECTRICAL
6,AWL,2.41,-42.22,189.03,67.02,389196.0,-150418.0,205891.0,485.00,-57.34,39.0,X-SC,7.48,81.0,-0.39,1.35,2.41,XY24,NTT,FMCG
40,ICICIPRULI,3.26,-29.99,23.47,-13.56,32368.0,-59091.0,137914.0,522.35,-55.17,40.0,X-MC,9.39,78.0,-1.83,0.90,3.94,X40,ATH,INSURANCE
27,GLAXO,-1.25,0.37,35.20,35.69,70406.0,732.0,200016.0,3380.16,-24.67,42.0,X-MC,9.73,56.0,0.01,1.31,18.88,X40,ATH,PHARMA
54,NESTLEIND,0.83,6.26,22.39,30.05,68180.0,17950.0,304509.0,1540.00,-20.17,49.0,X-LC,8.14,20.0,0.26,1.99,8.36,XY25,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,-1.90,-10.75,29.26,15.37,46708.0,-19225.0,159630.0,34390.26,-27.37,55.0,X-MC,1.33,58.0,-0.41,1.04,8.13,X40,ATH,PHARMA
17,CIPLA,0.15,5.55,17.38,23.89,55151.0,16672.0,317327.0,1648.14,19.62,65.0,X-LC,1.59,18.0,0.30,2.08,18.86,X40N,ATH,PHARMA
55,PAGEIND,-0.64,-7.35,43.40,32.85,107910.0,-19739.0,248640.0,50936.07,-31.08,50.0,X-MC,1.97,55.0,-0.18,1.63,18.73,X40,ATH,APPARELS
11,BERGEPAINT,4.10,-6.93,34.65,25.32,83117.0,-17874.0,239875.0,680.00,-19.10,71.0,X-MC,2.10,68.0,-0.22,1.57,25.79,XY24,NTT,PAINTS
8,BAJAJHLDNG,3.99,-2.93,30.81,26.99,93592.0,-9155.0,303772.0,14191.91,-11.97,52.0,X-LC,3.94,28.0,-0.10,1.99,26.31,X40,ATH,FINANCE


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,2.41,-42.22,189.03,67.02,389196.0,-150418.0,205891.0,485.00,-57.34,39.0,X-SC,7.48,81.0,-0.39,1.35,2.41,XY24,NTT,FMCG
40,ICICIPRULI,3.26,-29.99,23.47,-13.56,32368.0,-59091.0,137914.0,522.35,-55.17,40.0,X-MC,9.39,78.0,-1.83,0.90,3.94,X40,ATH,INSURANCE
56,PGHH,1.00,-47.33,148.76,31.02,167290.0,-101064.0,112456.0,17484.41,-61.22,50.0,X-SC,24.39,57.0,-0.60,0.74,5.87,X40,ATH,FMCG
35,HDFCLIFE,2.12,-5.80,49.21,40.56,144238.0,-18051.0,293108.0,811.41,-31.42,56.0,X-LC,10.08,31.0,-0.13,1.92,5.88,X40,ATH,INSURANCE
32,HAVELLS,-0.18,-31.61,92.17,31.42,206738.0,-103690.0,224301.0,2052.62,-41.11,51.0,X-MC,13.47,63.0,-0.50,1.47,5.91,X40,ATH,ELECTRICAL


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,1.00,-47.33,148.76,31.02,167290.0,-101064.0,112456.0,17484.41,-61.22,50.0,X-SC,24.39,57.0,-0.60,0.74,5.87,X40,ATH,FMCG
40,ICICIPRULI,3.26,-29.99,23.47,-13.56,32368.0,-59091.0,137914.0,522.35,-55.17,40.0,X-MC,9.39,78.0,-1.83,0.90,3.94,X40,ATH,INSURANCE
2,ACC,3.49,-45.10,218.42,74.81,321523.0,-120924.0,147204.0,3906.00,-45.40,61.0,X-SC,15.93,85.0,-0.38,0.96,9.05,XY24,BTT,CEMENT
37,HONAUT,1.71,-21.38,91.93,50.88,139366.0,-41238.0,151600.0,58192.80,-38.62,38.0,X-SC,6.67,90.0,-0.30,0.99,15.50,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-1.90,-10.75,29.26,15.37,46708.0,-19225.0,159630.0,34390.26,-27.37,55.0,X-MC,1.33,58.0,-0.41,1.04,8.13,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,-0.41,-20.60,62.09,28.70,215730.0,-90148.0,347447.0,452.00,-28.31,66.0,X-LC,8.05,1.0,-0.42,2.27,9.35,X40,NTT,FMCG
72,TCS,2.45,-24.83,68.62,26.75,251745.0,-121207.0,366868.0,4208.34,-39.28,63.0,X-LC,5.68,2.0,-0.48,2.40,27.24,X40,ATH,IT
45,INFY,3.47,-15.54,68.92,42.66,233326.0,-62309.0,338546.0,1972.00,-38.91,62.0,X-LC,8.70,3.0,-0.27,2.21,21.20,X40,NTT,IT
74,TMPV,5.46,-28.51,104.29,46.05,333560.0,-127533.0,319839.0,600.00,-85.99,62.0,X-LC,17.51,4.0,-0.38,2.09,12.44,XY24,NTT,AUTO
33,HCLTECH,1.82,-2.12,28.42,25.70,100418.0,-7639.0,353334.0,1837.07,-6.73,66.0,X-LC,5.83,8.0,-0.08,2.31,40.97,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,-2.33,-26.39,35.84,-0.01,33716.0,-33731.0,94074.0,2282.00,732.37,43.0,M-SC,19.93,231.0,-1.00,0.62,60.13,OX40N,BTT,AC
80,VALIANTORG,-14.84,-23.24,126.89,74.15,159347.0,-38026.0,125579.0,838.00,NaN,38.0,NaN,46.15,145.0,-0.24,0.82,86.88,XR,NTT,CHEMICALS
70,TANLA,-0.51,-25.89,190.34,115.17,381936.0,-70101.0,200660.0,1903.93,NaN,76.0,NaN,12.81,147.0,-0.18,1.31,77.38,AR,ATH,IT
25,FINCABLES,-1.48,48.73,38.80,106.44,85202.0,71950.0,219594.0,1629.94,8.16,41.0,M-SC,25.42,220.0,0.84,1.44,66.39,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.41,25.80,7.71,35.51,20328.0,54079.0,263653.0,1285.01,24.82,55.0,H-LC,11.55,120.0,2.66,1.72,38.75,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,-0.51,-25.89,190.34,115.17,381936.0,-70101.0,200660.0,1903.93,NaN,76.0,NaN,12.81,147.0,-0.18,1.31,77.38,AR,ATH,IT
71,TATAELXSI,4.39,-43.51,133.70,32.03,99580.0,-57355.0,74480.0,9161.00,-35.84,68.0,H-SC,13.79,158.0,-0.58,0.49,30.84,OX40N,NTT,IT
84,WIPRO,1.98,-8.82,111.69,93.01,254832.0,-22080.0,228160.0,420.00,-23.12,69.0,M-LC,8.03,98.0,-0.09,1.49,26.54,XR,NTT,IT
13,BSOFT,4.44,-40.12,145.81,47.20,122810.0,-56427.0,84226.0,811.90,NaN,62.0,NaN,11.36,172.0,-0.46,0.55,24.30,XR,ATH,IT
33,HCLTECH,1.82,-2.12,28.42,25.70,100418.0,-7639.0,353334.0,1837.07,-6.73,66.0,X-LC,5.83,8.0,-0.08,2.31,40.97,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.18
1,25,49.61
2,50,80.48


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.83
MC    20.58
SC    20.17
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.36
XY24     22.42
X40N     18.68
XR        9.78
XY25      9.46
OX40N     6.58
AR        4.95
SR        0.74
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.62
X-MC    18.20
H-SC     8.14
X-SC     7.04
M-SC     4.99
H-LC     1.72
M-LC     1.49
M-MC     1.06
H-MC     0.98
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.72,-26.67,72.14
IT,12.96,-29.36,96.87
FINANCE,10.62,-24.23,77.30
AUTO,8.52,-14.86,55.84
PHARMA,6.15,5.56,20.47
INSURANCE,5.87,-10.20,40.18
PAINTS,5.48,-23.24,44.87
MISC,5.44,-48.20,107.76
BANKS,5.01,-17.18,75.95


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4004615.0,19
X40,2210779.0,17
XR,1559694.0,13
X40N,1481930.0,12
AR,1035969.0,7
OX40N,786115.0,10
XY25,765962.0,6
SR,299851.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2883872.0,19
X-MC,1623549.0,14
X-SC,1198968.0,6
H-SC,906780.0,8
M-SC,401743.0,6
H-MC,264490.0,1
M-LC,254832.0,1
M-MC,82632.0,1
L-MC,50649.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1288983.0      8
X-SC       XY24       710719.0      2
X-LC       X40N       562562.0      5
H-SC       XY24       560406.0      3
X-LC       XY24       555782.0      2
X-MC       X40N       536286.0      4
           X40        529298.0      6
X-LC       XY25       378234.0      3
X-MC       XY25       305096.0      2
X-SC       X40        271453.0      2
H-MC       XY24       264490.0      1
M-LC       XR         254832.0      1
X-MC       XY24       252869.0      2
H-SC       OX40N      240184.0      3
X-SC       X40N       216796.0      2
M-SC       OX40N      210538.0      4
           XY24       131581.0      1
H-SC       XR         106190.0      2
X-LC       XR          98311.0      1
M-MC       XY25        82632.0      1
M-SC       XR          59624.0      1
L-MC       XR          50649.0      1
H-LC       AR          20328.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 38.0 seconds
